# Approximate inference: Gibbs sampling and particle filtering

_Artificial Intelligence — more_

**Too big to compute exactly? Draw many samples and let the histogram be your answer.**

Exact inference is too slow on big, tangled networks. Approximate inference trades exactness for speed by sampling.

     Gibbs sampling walks through the variables, resampling one at a time from its conditional given all the others. The visited states form samples from the posterior.

---

This notebook builds Gibbs sampling from scratch, one step at a time. Run each cell, read the note above it, and watch the sampled histogram converge to the true posterior. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## Reference implementation — Python + NumPy

We'll run Gibbs sampling on the smallest interesting case: a joint distribution over two binary variables `X` and `Y`. We build it in three steps: (1) the unnormalized joint table and its exact posterior, (2) the conditional sampler, (3) the Gibbs loop that resamples one variable at a time.

### Step 1 — Write down the joint and its exact answer

`phi` is an unnormalized 2x2 table: one entry for each combination of `X` and `Y` in `{0, 1}`. Because this case is tiny, we can normalize the whole table directly to get the **true posterior** — the answer Gibbs sampling should converge to. We keep it so we can measure the sampler's error at the end.

In [ ]:
rng = np.random.default_rng(0)

# Unnormalized joint over (X, Y), each in {0, 1}: a 2x2 table.
phi = np.array([[1.0, 2.0],
                [3.0, 4.0]])

# Tiny enough to normalize exactly — this is the target Gibbs should recover.
true_post = phi / phi.sum()

### Step 2 — The conditional distribution of one variable given the other

Gibbs sampling never touches the full joint while running — it only needs the distribution of **one** variable conditioned on the current value of the other. To get `P(X | Y=y)` we pick the row (or column) of `phi` that fixes the other variable, then renormalize it to sum to 1.

In [ ]:
def cond(fix_axis, fix_val):
    # Slice phi so the variable on `fix_axis` is held at `fix_val`,
    # then renormalize the remaining row/column into a probability.
    if fix_axis == 0:
        row = phi[fix_val, :]   # fix X -> distribution over Y
    else:
        row = phi[:, fix_val]   # fix Y -> distribution over X
    return row / row.sum()

### Step 3 — Walk the chain, resampling one variable at a time

Starting from `(0, 0)`, each iteration resamples `X` from `P(X | Y)`, then resamples `Y` from `P(Y | X)` using the freshly drawn `X`. We tally how often the chain visits each `(x, y)` state. After 20000 steps the visit frequencies (`emp`) approximate the true posterior — and we print the largest gap to confirm.

In [ ]:
x = 0
y = 0
counts = np.zeros((2, 2))

for t in range(20000):
    x = rng.choice(2, p=cond(1, y))   # resample X | Y
    y = rng.choice(2, p=cond(0, x))   # resample Y | X
    counts[x, y] += 1

emp = counts / counts.sum()           # empirical posterior from visit frequencies

print("true posterior:\n", np.round(true_post, 3))
print("Gibbs estimate:\n", np.round(emp, 3))
print("max abs error:", round(float(np.max(np.abs(emp - true_post))), 4))

## Visualize the data & results

_Sprinkler-Rain network: does Gibbs sampling recover the joint posterior over (Sprinkler, Rain) given wet grass?_

### Step 1 — Define the Sprinkler-Rain-WetGrass network

This is the classic Bayes net: a hidden cause `C` (cloudy) drives both the Sprinkler `S` and the Rain `R`, and either one can make the grass wet `W`. Each array is a conditional probability table — for example `P_S_C[c]` gives `P(Sprinkler | Cloudy=c)`. We will condition on `WetGrass=True` throughout.

In [ ]:
rng = np.random.default_rng(0)

# Conditional probability tables for the Sprinkler-Rain-WetGrass net.
P_C = np.array([0.5, 0.5])                       # P(Cloudy)
P_S_C = np.array([[0.5, 0.5], [0.9, 0.1]])       # P(Sprinkler | Cloudy)
P_R_C = np.array([[0.8, 0.2], [0.2, 0.8]])       # P(Rain | Cloudy)

P_W = np.zeros((2, 2, 2))                         # P(Wet | Sprinkler, Rain)
P_W[0, 0] = [1.0, 0.0]
P_W[0, 1] = [0.1, 0.9]
P_W[1, 0] = [0.1, 0.9]
P_W[1, 1] = [0.01, 0.99]

# Helper: renormalize a vector of scores into a probability distribution.
norm = lambda v: v / v.sum()

### Step 2 — Gibbs-sample the hidden variables given wet grass

With `WetGrass` fixed to True, we resample each hidden variable from its conditional given the others. Each conditional is the product of the factors that mention that variable — for `C` that is its prior times `P(S|C)` and `P(R|C)`; for `S` and `R` it also includes the `WetGrass=True` likelihood. We tally the visited `(S, R)` states over 40000 steps.

In [ ]:
C = 0
S = 0
R = 0
counts = np.zeros((2, 2))

for t in range(40000):
    # Resample Cloudy given Sprinkler and Rain.
    score_C = np.array([P_C[c] * P_S_C[c, S] * P_R_C[c, R] for c in range(2)])
    C = rng.choice(2, p=norm(score_C))

    # Resample Sprinkler given Cloudy and the wet-grass evidence.
    score_S = np.array([P_S_C[C, s] * P_W[s, R, 1] for s in range(2)])
    S = rng.choice(2, p=norm(score_S))

    # Resample Rain given Cloudy and the wet-grass evidence.
    score_R = np.array([P_R_C[C, r] * P_W[S, r, 1] for r in range(2)])
    R = rng.choice(2, p=norm(score_R))

    counts[S, R] += 1

emp = (counts / counts.sum()).flatten()

### Step 3 — Plot the recovered posterior

Each bar is one of the four `(Sprinkler, Rain)` combinations, with its estimated probability given wet grass. The tallest bars reveal the most likely explanations for the wet grass under the model.

In [ ]:
labels = ["S=off, R=no", "S=off, R=rain", "S=on, R=no", "S=on, R=rain"]

fig, ax = plt.subplots()
bars = ax.bar(labels, emp, color="#4ea1ff")
for b, v in zip(bars, emp):
    ax.text(b.get_x() + b.get_width() / 2, v, str(round(v, 3)), ha="center", va="bottom")
ax.set_title("Gibbs estimate of P(Sprinkler, Rain | WetGrass) (40k samples)")
plt.show()